# Imports

In [ ]:
# --- Cell: imports ---
import os
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

import sys
sys.path.append('.')  # folder containing vti_magnetic_import.py
from vti_magnetic_import import load_vti_sample, inspect_vti

from scipy.ndimage import gaussian_filter, map_coordinates
from skimage.measure import marching_cubes
import pyvista as pv

import matplotlib.ticker as mticker
import pandas as pd

import json

import Functions_3D as vis
import error as er

# Load Volumes

In [ ]:
def load_sample_data(save_dir, sample_name, date):
    """
    Load back the cropped volume for one sample
    """
    volume_path = os.path.join(save_dir, f'{sample_name}_volume_{date}.npz')

    npz = np.load(volume_path)
    volume = {
        'Mx': npz['Mx'], 'My': npz['My'], 'Mz': npz['Mz'],
        'magnitude': npz['magnitude'],
        'shape': tuple(npz['shape']),
    }

    spacing = tuple(npz['spacing'])
    z_offset = 0

    return volume, spacing, z_offset

In [ ]:
# # sample_names = ['452', '452', '452', '665'] 
# sample_colors = ['limegreen', 'deeppink', 'cyan', 'gold']
# sample = 1


# # sample_name = sample_names[sample]
# color = sample_colors[sample]

# dates = ['ground_truth_5vx_99px', 'sim_stripe_noreg_2px_5z_NOV26_v2','sim_stripe_TV_2px_5z_NOV26_v2', '452_volume_sim_stripe_noreg_2px_5z_180tomo_thick', 'stripes_recon_magconst_2_5z_real_angles_200iter', 'stripes_recon_noreg_2_5z_real_angles', 'stripes_recon_magconst_2_5z_real_angles', 'stripes_recon_TV_2_5z_real_angles_200iter']
# ct = 0

# for date in dates: 
#     print(date)

#     data_dir = r'C:\Users\emmab\Projects\Vector_Tomography\stripe_dw_2_5z'

#     volume, spacing, ___ = load_sample_data(data_dir, '452', date)
#     print(spacing)

#     if ct == 0: 
#         original = volume
#     elif ct == 1: 
#         normalized = volume
#     elif ct == 2: 
#         mag_const = volume

#     ct = ct + 1

    

In [ ]:
data_dir = r'C:\Users\emmab\Projects\Vector_Tomography\data_dir\452'
sample_name = '452'

# short name -> file identifier (the `date` string in your filenames)
# volume_files = {
#     'ground_truth':          'ground_truth_5vx_99px',
#     'sim_noreg':             'sim_stripe_noreg_2px_5z_NOV26_v2',
#     'sim_TV':                'sim_stripe_TV_2px_5z_NOV26_v2',
#     'sim_noreg_180tomo':     'sim_stripe_noreg_2px_5z_180tomo_thick', 
#     'real_magconst_200iter': 'stripes_recon_magconst_2_5z_real_angles_200iter',
#     'real_noreg':            'stripes_recon_noreg_2_5z_real_angles',
#     'real_magconst':         'stripes_recon_magconst_2_5z_real_angles',
#     'real_TV_200iter':       'stripes_recon_TV_2_5z_real_angles_200iter',
# }


# # Bloch
# volume_files = {
#     'ground_truth':      'ground_truth_5vx_99px',
#     'crop_from_15vox':   'TV_real_angle_cropped',
#     'recon_5vox':        'TV_recon5',
#     'mask_5vox':         'TV_real_angle_mask', 

# }


# # Neel
# volume_files = {
#     'ground_truth':      'ground_truth_5vx_99px',
#     'crop_from_15vox':   'Neel_TV_real_angle_cropped',
#     'recon_5vox':        'TV_recon5',
#     'mask_5vox':         'Neel_TV_real_angle_mask', 

# }


# # hybrid
# volume_files = {
#     'ground_truth':      'ground_truth_5vx_99px',
#     'crop_from_15vox':   'hybrid_TV_real_angle_cropped',
#     'recon_5vox':        'TV_recon5',
#     'mask_5vox':         'hybrid_TV_real_angle_masked', 

# }

# reprojected 452
volume_files = {
    # 'ground_truth':      'ground_truth_reproject_99px',
    # 'crop_from_15vox':   'TV_15thickcrop_reproject',
    'ground_truth':        'TV_5thick_reproject',
    'mask_5vox':         'TV_mask_reproject', 

}

# --- crop settings ---
CROP_XY = True          # flag: turn cropping on/off
CROP_TRIGGER_SIZE = 101 # crop any volume whose (ny, nx) equals this
CROP_N_PIX = 1          # voxels removed from each x/y edge (set this to what you need)

volumes = {}
spacings = {}

for name, file_id in volume_files.items():
    volume, spacing, _ = load_sample_data(data_dir, sample_name, file_id)

    nz, ny, nx = volume['shape']
    if CROP_XY and (ny == CROP_TRIGGER_SIZE or nx == CROP_TRIGGER_SIZE):
        volume = vis.crop_xy_edges(volume, n_pix=CROP_N_PIX, verbose=False)
        print(f"{name}: cropped {(nz, ny, nx)} -> {volume['shape']}")

    volumes[name] = volume
    spacings[name] = spacing
    print(f"{name:25s} shape={volume['shape']}  spacing={spacing}")

gt = volumes['ground_truth']

components = ['Mx', 'My', 'Mz', 'magnitude']
COMPS3 = ['Mx', 'My', 'Mz']
out_dir = os.path.join(data_dir, 'comparison_results')
os.makedirs(out_dir, exist_ok=True)


recon_names = []
for n in volumes:
    if n == 'ground_truth':
        continue
    if volumes[n]['shape'] != gt['shape']:
        print(f"WARNING: skipping {n}: shape {volumes[n]['shape']} != GT {gt['shape']}")
        continue
    recon_names.append(n)

# Visualize inputs

In [ ]:
# vis.plot_three_slice_fancy(original, field='Mx', label = 'original')
# vis.plot_three_slice_fancy(normalized, field='Mx', label = 'normalized')
# vis.plot_three_slice_fancy(mag_const, field='Mx', label = 'mag_const')

In [ ]:
def plot_residual_fancy(sample1, sample2, field='Mz', label1 = None, label2 = None, x_index=None, y_index=None, z_index=None,
                      cmap='viridis', vmin=None, vmax=None):

    field_aliases = {'mag': 'magnitude', 'magnitude': 'magnitude',
                      'mx': 'Mx', 'my': 'My', 'mz': 'Mz'}
    key = field_aliases.get(field.lower())
    if key is None or key not in sample1:
        raise KeyError(
            f"field '{field}' not recognized/available. "
            f"Expected one of: Mx, My, Mz, magnitude."
        )

    if field == 'Mx': 
        data1 = sample1[key]   # shape (nz, ny, nx)
        nz, ny, nx = sample1['shape']

        data2 = sample2[key]  # shape (nz, ny, nx)

    else: 
        data1 = sample1[key] / np.nanmax(sample1[key])   # shape (nz, ny, nx)
        nz, ny, nx = sample1['shape']

        data2 = sample2[key] / np.max(sample2[key])  # shape (nz, ny, nx)

    if x_index is None:
        x_index = nx // 2
    if y_index is None:
        y_index = ny // 2
    if z_index is None:
        z_index = nz // 2


    R_x = abs(data1[:, :, x_index] - data2[:, :, x_index])/((data1[:, :, x_index] + data2[:, :, x_index])/2) *100
    R_y = abs(data1[:, y_index, :] - data2[:, y_index, :])/((data1[:, y_index, :] + data2[:, y_index, :])/2) *100
    R_z = abs(data1[z_index, :, :] - data2[z_index, :, :])/((data1[z_index, :, :] + data2[z_index, :, :])/2) *100
    # R_y = data1[:, y_index, :] - data2[:, y_index, :]
    # R_z = data1[z_index, :, :] - data2[z_index, :, :]

    LABEL_FONT = {'family': 'Times New Roman', 'weight': 'bold', 'size': 16}
    TITLE_FONT = {'family': 'Times New Roman', 'weight': 'bold', 'size': 20}
    TICK_FONT = {'family': 'Times New Roman', 'size': 13}
    CBAR_LABEL_FONT = {'family': 'Times New Roman', 'weight': 'bold', 'size': 20}

    if field == 'Mz':
        y_lim = R_z

    elif field == 'Mx':
        y_lim = R_x

    elif field == 'My': 
        y_lim = R_y

    #print(R_x)
    
    if vmin is None:
        vmin = np.nanmin(y_lim)
    if vmax is None:
        vmax = np.nanmax(y_lim)


    def style_axis(ax):
        ax.set_xlabel(ax.get_xlabel(), fontdict=LABEL_FONT)
        ax.set_ylabel(ax.get_ylabel(), fontdict=LABEL_FONT)
        for label in ax.get_xticklabels() + ax.get_yticklabels():
            label.set_fontname('Times New Roman')
            label.set_fontsize(TICK_FONT['size'])

    def style_colorbar(cbar, label_text):
        cbar.set_label(label_text, fontdict=CBAR_LABEL_FONT)
        cbar.formatter = mticker.ScalarFormatter(useMathText=True)
        cbar.formatter.set_powerlimits((0, 0))
        cbar.update_ticks()
        for label in cbar.ax.get_yticklabels():
            label.set_fontname('Times New Roman')
            label.set_fontsize(TICK_FONT['size'])
        cbar.ax.yaxis.get_offset_text().set_fontname('Times New Roman')
        cbar.ax.yaxis.get_offset_text().set_fontsize(TICK_FONT['size'])

    # Conversion factor: 1 pix = 8 nm
    PIX_TO_NM = 2 #8

    # Calculate extents in nm
    extent_x = [0, ny * PIX_TO_NM / 1000, 0, nz * PIX_TO_NM]  # y, z
    extent_y = [0, nx * PIX_TO_NM / 1000, 0, nz * PIX_TO_NM]  # x, z
    extent_z = [0, nx * PIX_TO_NM / 1000, 0, ny * PIX_TO_NM / 1000]  # x, y

    # Create figure with GridSpec for custom layout
    fig = plt.figure(figsize=(25,8)) ##################################################################################
    gs = fig.add_gridspec(2, 3, width_ratios=[1, 2, 0.15], height_ratios=[1, 1], 
                        hspace=0.37, wspace=0.15)

    # Z slice (left, full square)
    ax_z = fig.add_subplot(gs[:, 0])
    im_z = ax_z.imshow(R_z, origin='lower', aspect='equal', cmap=cmap,
                        vmin=vmin, vmax=vmax, extent=extent_z)
    ax_z.set_title(f"Z Residual ", fontdict=TITLE_FONT)
    ax_z.set_xlabel('x (μm)', fontdict=LABEL_FONT)
    ax_z.set_ylabel('y (μm)', fontdict=LABEL_FONT)
    ax_z.axhline(y_index * PIX_TO_NM/1000, color='magenta', lw=5, linestyle='--', zorder=3)
    ax_z.axvline(x_index * PIX_TO_NM/1000, color='darkorange', lw=5, linestyle='--', zorder=3)
    style_axis(ax_z)

    # X slice (top right, thin and long)
    ax_x = fig.add_subplot(gs[0, 1])
    im_x = ax_x.imshow(R_x, origin='lower', aspect='auto', cmap=cmap,
                        vmin=vmin, vmax=vmax, extent=extent_x)
    ax_x.set_title(f"X Residual ", fontdict=TITLE_FONT, color='darkorange')
    ax_x.set_xlabel('y (μm)', fontdict=LABEL_FONT)
    ax_x.set_ylabel('z (nm)', fontdict=LABEL_FONT)
    style_axis(ax_x)

    # Y slice (bottom right, thin and long)
    ax_y = fig.add_subplot(gs[1, 1])
    im_y = ax_y.imshow(R_y, origin='lower', aspect='auto', cmap=cmap,
                        vmin=vmin, vmax=vmax, extent=extent_y)
    ax_y.set_title(f"Y Residual ", fontdict=TITLE_FONT, color='magenta')
    ax_y.set_xlabel('x (μm)', fontdict=LABEL_FONT)
    ax_y.set_ylabel('z (nm)', fontdict=LABEL_FONT)
    style_axis(ax_y)


    # Colorbar for Z slice
    cbar_xy = fig.colorbar(im_x, ax=[ax_x, ax_y], orientation='vertical', 
                            pad=0.05, aspect=30)
    style_colorbar(cbar_xy, key)

    fig.suptitle(f"{key} Contrast at Volume Center \n {label1} - {label2}", 
                fontfamily=LABEL_FONT['family'],
                fontweight=LABEL_FONT['weight'],
                fontsize=20)

    plt.show()
    
    return 

In [ ]:


# plot_residual_fancy(original, normalized, field='My', label1 = 'original', label2 = 'normalized', x_index=None, y_index=None, z_index=None,
#                       cmap='Reds', vmin=0, vmax=100)

# plot_residual_fancy(original, mag_const, field='My', label1 = 'original', label2 = 'mag_const', x_index=None, y_index=None, z_index=None,
#                       cmap='Reds', vmin=0, vmax=100)

# # plot_residual_fancy(normalized, mag_const, field='Mx',label1 = 'normalized', label2 = 'mag_const',  x_index=None, y_index=None, z_index=None,
# #                       cmap='seismic', vmin=None, vmax=None)


# From claude, how to handle and plot errors for large amount of simulations

In [ ]:


# # Global reference scale and a "sample" mask (voxels where GT has signal)
# MASK_FRAC = 0.1   # mask = GT magnitude > 10% of its max
# ref_scale = np.nanmax(gt['magnitude'])
# mask = gt['magnitude'] > MASK_FRAC * ref_scale
# print(f"ref_scale = {ref_scale:.4g}, mask covers {100 * mask.mean():.1f}% of voxels")

# def error_metrics(recon, truth, region=None):
#     diff = recon - truth
#     if region is not None:
#         diff = diff[region]
#     rmse = np.sqrt(np.nanmean(diff**2))
#     return {
#         'rmse': rmse,
#         'nrmse_pct': 100 * rmse / ref_scale,       # RMSE as % of GT full scale
#         'mae': np.nanmean(np.abs(diff)),
#         'bias': np.nanmean(diff),                   # signed: systematic over/under-estimate
#         'max_abs_err': np.nanmax(np.abs(diff)),
#     }

# rows = []
# for n in recon_names:
#     for comp in components:
#         for region_name, region in [('full', None), ('masked', mask)]:
#             m = error_metrics(volumes[n][comp], gt[comp], region)
#             rows.append({'volume': n, 'component': comp, 'region': region_name,
#                          'spacing': spacings[n], **m})

# df = pd.DataFrame(rows)
# df.to_csv(os.path.join(out_dir, 'metrics_vs_ground_truth.csv'), index=False)

# # Quick-look table: NRMSE (%) in the masked region, volumes x components
# summary = (df[df.region == 'masked']
#            .pivot(index='volume', columns='component', values='nrmse_pct')[components])
# print(summary.round(2))

In [ ]:
def _draw_grid(maps, names, cmap, vmin, vmax, cbar_label, title_fn, fname,
               col_width=2.4, font='Times New Roman', save=True):
    nz, ny, nx = gt['shape']
    zi, yi, xi = nz // 2, ny // 2, nx // 2
    n = len(names)
    row_titles = [f'XY (z={zi})', f'XZ (y={yi})', f'YZ (x={xi})']

    cm = plt.get_cmap(cmap).copy()
    cm.set_bad('lightgray')          # NaN (outside mask) shows as gray

    with plt.rc_context({'font.family': font}):
        fig = plt.figure(figsize=(col_width * n + 0.8, 7.5))
        gs = fig.add_gridspec(4, n, height_ratios=[1, 0.45, 0.45, 0.07],
                              hspace=0.12, wspace=0.05)
        for j, k in enumerate(names):
            m = maps[k]
            slices = [m[zi, :, :], m[:, yi, :], m[:, :, xi]]
            for i, s in enumerate(slices):
                ax = fig.add_subplot(gs[i, j])
                im = ax.imshow(s, origin='lower', cmap=cm, vmin=vmin, vmax=vmax,
                               aspect='equal' if i == 0 else 'auto')
                ax.set_xticks([]); ax.set_yticks([])
                if j == 0:
                    ax.set_ylabel(row_titles[i], fontsize=13, fontweight='bold')
                if i == 0:
                    ax.set_title(title_fn(k, m), fontsize=11, fontweight='bold')

        c0, c1 = int(n * 0.2), max(int(np.ceil(n * 0.8)), int(n * 0.2) + 1)
        cax = fig.add_subplot(gs[3, c0:c1])
        cbar = fig.colorbar(im, cax=cax, orientation='horizontal')
        cbar.set_label(cbar_label, fontsize=13, fontweight='bold')
        cbar.ax.tick_params(labelsize=11)
        if save:
            fig.savefig(os.path.join(out_dir, fname), dpi=300, bbox_inches='tight')
        plt.show()

def _rms(d):
    d = d[np.isfinite(d)]
    return np.sqrt(np.mean(d**2)) if d.size else np.nan

def _mean(d):
    d = d[np.isfinite(d)]
    return d.mean() if d.size else np.nan


def angle_deg_map(n):
    """Per-voxel angle (deg) between r_hat and g_hat for volume n; NaN where invalid."""
    # theta = 2*arcsin(|r_hat - g_hat| / 2) is identical to arccos(r_hat . g_hat)
    # but stays accurate for small angles (arccos loses precision near cos = 1).
    chord = np.linalg.norm(nhat_res[n]['resid'], axis=-1)
    return np.degrees(2 * np.arcsin(np.clip(chord / 2, 0, 1)))

In [ ]:
# ---- reference quantities: no masking, no thresholds ----
EPS = 1e-12   # only guards against dividing by (numerically) zero

def stack_vec(v):
    return np.stack([v['Mx'], v['My'], v['Mz']], axis=-1)    # (nz, ny, nx, 3)

gt_vec = stack_vec(gt)
gt_mag = np.linalg.norm(gt_vec, axis=-1)

gt_ok = np.isfinite(gt_mag) & (gt_mag > EPS)
gt_hat = np.full_like(gt_vec, np.nan, dtype=float)
gt_hat[gt_ok] = gt_vec[gt_ok] / gt_mag[gt_ok][:, None]


def unit_resid(recon):
    """Unit-vector residual (recon_hat - gt_hat), shape (nz,ny,nx,3)"""
    rv = stack_vec(recon)
    rmag = np.linalg.norm(rv, axis=-1)
    gmag = np.linalg.norm(gt_vec, axis=-1)
    r_hat = rv / rmag[..., None] 
    g_hat = gt_vec / gmag[..., None] 
    return {'r_hat': r_hat, 'g_hat': g_hat, 'resid': r_hat - g_hat}


nhat_res = {n: unit_resid(volumes[n]) for n in recon_names}


# ---- plots ----
def plot_unit_resid_grid(comp, names, lim=None, pct=99.5, **kw):
    i = ['Mx', 'My', 'Mz'].index(comp)
    maps = {k: nhat_res[k]['resid'][..., i] for k in names}
    if lim is None:
        lim = max(np.nanpercentile(np.abs(m), pct) for m in maps.values())
    _draw_grid(maps, names, 'RdBu_r', -lim, lim,
               f'unit {comp} residual (recon − GT)',
               lambda k, m: f'{k}\nRMS = {np.sqrt(np.nanmean(m**2)):.3f}',
               f'unit_residual_{comp}_rows_xyz.png', **kw)


def plot_unit_component_grid(comp, names, include_gt=True, **kw):
    """Recon unit-vector component, with GT as the first column for reference."""
    i = ['Mx', 'My', 'Mz'].index(comp)
    maps = {k: nhat_res[k]['r_hat'][..., i] for k in names}
    cols = list(names)
    if include_gt:
        maps = {'ground_truth': gt_hat[..., i], **maps}
        cols = ['ground_truth'] + cols
    _draw_grid(maps, cols, 'viridis', -1, 1,
               f'unit {comp}',
               lambda k, m: k,
               f'unit_rhat_{comp}_rows_xyz.png', **kw)




In [ ]:

# Residuals on one shared scale so Mx leakage is comparable to My/Mz error
lim_u = max(np.nanpercentile(np.abs(nhat_res[k]['resid'][..., i]), 99.5)
            for k in recon_names for i in range(3))
for comp in ['Mx', 'My', 'Mz']:
    plot_unit_component_grid(comp, recon_names)
    plot_unit_resid_grid(comp, recon_names, lim=2)

# Save for later plotting / the z-slice step
# for n in recon_names:
#     np.savez_compressed(os.path.join(out_dir, f'unit_vectors_{n}.npz'), **nhat_res[n])
# np.save(os.path.join(out_dir, 'unit_vectors_ground_truth.npy'), gt_hat)

In [ ]:
# ---- settings ----
PLOT_SLICES = False    # flag: plot residual maps per z slice
Z_INDEX = None        # None -> prompt for indices; or an int, a list, or 'all'

nz = gt['shape'][0]

def ask_z_indices(nz):
    s = input(f"z index to plot (0-{nz-1}, comma-separated, or 'all'): ").strip().lower()
    if s in ('all', 'a', ''):
        return list(range(nz))
    idx = [int(t) for t in s.replace(' ', '').split(',') if t != '']
    bad = [i for i in idx if not 0 <= i < nz]
    if bad:
        raise ValueError(f"z indices out of range 0-{nz-1}: {bad}")
    return idx

def resolve_z(z_index, nz):
    if z_index is None:
        return ask_z_indices(nz)
    if z_index == 'all':
        return list(range(nz))
    return [z_index] if isinstance(z_index, int) else list(z_index)


# ---- 1. per-slice table: mean (signed) unit-vector residual per component ----
rows = []
for n in recon_names:
    res = nhat_res[n]['resid']                       # (nz, ny, nx, 3)
    cosang = np.sum(nhat_res[n]['r_hat'] * nhat_res[n]['g_hat'], axis=-1)
    ang = np.degrees(np.arccos(np.clip(cosang, -1, 1)))   # NaN where invalid
    for z in range(nz):
        row = {'volume': n, 'z': z}
        for i, c in enumerate(COMPS3):
            row[f'u{c}_mean'] = _mean(res[z][..., i])            # signed
            row[f'u{c}_mean_abs'] = _mean(np.abs(res[z][..., i]))
            row[f'u{c}_rms'] = _rms(res[z][..., i])
        row['ang_mean_deg'] = _mean(ang[z])
        rows.append(row)

slice_resid_df = pd.DataFrame(rows)
slice_resid_df.to_csv(os.path.join(out_dir, 'unit_resid_per_z_slice.csv'), index=False)

# Tables: rows = recon condition, columns = z slice
for c in COMPS3:
    print(f"\nMean signed unit-{c} residual (recon - GT) per z slice")
    print(slice_resid_df.pivot(index='volume', columns='z', values=f'u{c}_mean').round(4))
print("\nMean angular error (deg) per z slice")
print(slice_resid_df.pivot(index='volume', columns='z', values='ang_mean_deg').round(2))



# ---- 2. optional residual maps for chosen slices ----
def plot_slice_residuals(z_index, names, comps=COMPS3, lim=None, pct=99.5,
                         cell=2.6, font='Times New Roman', save=True):
    """Rows = components, columns = recon conditions, one figure per z slice."""
    zs = resolve_z(z_index, nz)
    if lim is None:   # one scale for every slice/volume/component so slices are comparable
        lim = max(np.nanpercentile(np.abs(nhat_res[k]['resid'][..., i]), pct)
                  for k in names for i in range(3))
    cm = plt.get_cmap('RdBu_r').copy()
    cm.set_bad('lightgray')
    n = len(names)

    for z in zs:
        with plt.rc_context({'font.family': font}):
            fig, axes = plt.subplots(len(comps), n, squeeze=False,
                                     figsize=(cell * n + 0.8, cell * len(comps) + 1.2))
            for r, c in enumerate(comps):
                i = COMPS3.index(c)
                for j, k in enumerate(names):
                    d = nhat_res[k]['resid'][z][..., i]
                    ax = axes[r, j]
                    im = ax.imshow(d, origin='lower', cmap=cm, vmin=-lim, vmax=lim,
                                   aspect='equal')
                    ax.set_xticks([]); ax.set_yticks([])
                    ax.set_xlabel(f'mean={_mean(d):+.3f}  rms={_rms(d):.3f}', fontsize=9)
                    if r == 0:
                        ax.set_title(k, fontsize=11, fontweight='bold')
                    if j == 0:
                        ax.set_ylabel(f'unit {c}', fontsize=13, fontweight='bold')
            cbar = fig.colorbar(im, ax=axes, orientation='horizontal',
                                fraction=0.04, pad=0.1)
            cbar.set_label('unit-vector residual (recon − GT, direction cosine)',
                           fontsize=12, fontweight='bold')
            fig.suptitle(f'z slice {z}', fontsize=14, fontweight='bold')
            if save:
                fig.savefig(os.path.join(out_dir, f'unit_resid_z{z}.png'),
                            dpi=300, bbox_inches='tight')
            plt.show()

if PLOT_SLICES:
    plot_slice_residuals(Z_INDEX, recon_names)

In [ ]:
vol_colors = {n: plt.cm.tab10(i % 10) for i, n in enumerate(recon_names)}

def rms_vs_z_table(names=None):
    """Per-z-slice RMS of the unit-vector residual (components, vector) and angle stats."""
    names = names or recon_names
    rows = []
    for n in names:
        res = nhat_res[n]['resid']                         # (nz, ny, nx, 3)
        ang = angle_deg_map(n)                             # (nz, ny, nx)
        for z in range(res.shape[0]):
            row = {'volume': n, 'z': z}
            for i, c in enumerate(COMPS3):
                row[c] = _rms(res[z][..., i])
            row['vector'] = _rms(np.linalg.norm(res[z], axis=-1))
            row['angle'] = _rms(ang[z])                    # RMS angle, degrees
            row['angle_mean_deg'] = _mean(ang[z])          # mean angle, degrees
            rows.append(row)
    return pd.DataFrame(rows)


def plot_rms_vs_z(names=None, which=('Mx', 'My', 'Mz'), sharey=False,
                  show_baseline=True, font='Times New Roman', save=False, title = 'Bloch'):
    """
    One panel per entry in `which`; every volume is a line in each panel.
    show_baseline draws the RMS expected if recon directions were random
    (0.82 per component, 1.41 for the full vector).
    """
    names = names or recon_names
    df = rms_vs_z_table(names)
    df.to_csv(os.path.join(out_dir, 'unit_resid_rms_vs_z.csv'), index=False)

    baseline = {'Mx': np.sqrt(2/3), 'My': np.sqrt(2/3), 'Mz': np.sqrt(2/3),
                'vector': np.sqrt(2)}
    nz_ = int(df['z'].max()) + 1

    with plt.rc_context({'font.family': font}):
        fig, axes = plt.subplots(1, len(which), figsize=(4.2 * len(which) + 0.5, 4.4),
                                 sharey=sharey, squeeze=False)
        for ax, w in zip(axes[0], which):
            for n in names:
                d = df[df.volume == n].sort_values('z')
                ax.plot(d['z'], d[w], marker='o', lw=2,
                        color=vol_colors.get(n), label=n)
            if show_baseline:
                ax.axhline(baseline[w], color='k', ls='--', lw=1, alpha=0.6)
            ax.set_title(f'unit {w}' if w != 'vector' else 'unit vector',
                         fontsize=14, fontweight='bold')
            ax.set_xlabel('z slice index', fontsize=12, fontweight='bold')
            ax.set_xticks(range(nz_))
            ax.grid(alpha=0.3)
        axes[0, 0].set_ylabel('RMS of unit-vector residual', fontsize=12,
                              fontweight='bold')
        h, l = axes[0, 0].get_legend_handles_labels()
        fig.legend(h, l, loc='lower center', ncol=min(len(l), 4), fontsize=10,
                   frameon=False, bbox_to_anchor=(0.5, -0.1))
        fig.suptitle(title, fontsize = 20)
        fig.tight_layout()
        if save:
            fig.savefig(os.path.join(out_dir, 'unit_resid_rms_vs_z.png'),
                        dpi=300, bbox_inches='tight')
        plt.show()
    return df


rms_z_df = plot_rms_vs_z(title= '452 reprojected')


In [ ]:
stop

In [ ]:
nz = gt['shape'][0]

nhat_res = {n: unit_resid(volumes[n]) for n in recon_names}
for n in recon_names:
    np.savez_compressed(os.path.join(out_dir, f'unit_vectors_{n}.npz'),
                        **nhat_res[n])

# Table: RMS and signed bias of each unit-vector component
rows = []
for n in recon_names:
    row = {'volume': n}
    for i, c in enumerate(['Mx', 'My', 'Mz']):
        d = nhat_res[n]['resid'][..., i]
        row[f'u{c}_rms'] = np.sqrt(np.nanmean(d**2))
        row[f'u{c}_bias'] = np.nanmean(d)
    rows.append(row)
unit_df = pd.DataFrame(rows).set_index('volume')
unit_df.to_csv(os.path.join(out_dir, 'unit_vector_component_errors.csv'))
print(unit_df.round(3))

# Shared scale across components so Mx leakage is comparable to My/Mz error
lim_u = max(np.nanpercentile(np.abs(nhat_res[k]['resid'][..., i]), 99.5)
            for k in recon_names for i in range(3))
for comp in ['Mx', 'My', 'Mz']:
    plot_unit_resid_grid(comp, recon_names, lim=lim_u)
    plot_unit_component_grid(comp, recon_names)

In [ ]:
AS_PERCENT = True   # True: residuals in % of GT full scale; False: raw data units


def _unit_scale(as_percent):
    """Multiplier applied to residuals: 100/ref_scale for percent, 1 for raw."""
    if as_percent is None:
        as_percent = AS_PERCENT
    return (100.0 / ref_scale if as_percent else 1.0), as_percent


def global_residual_lim(names, comps=('Mx', 'My', 'Mz'), pct=99.5, as_percent=None):
    """One symmetric color limit shared across components/volumes, in the displayed units."""
    scale, _ = _unit_scale(as_percent)
    return scale * max(
        np.nanpercentile(np.abs(volumes[k][c] - gt[c]), pct)
        for c in comps for k in names
    )


def plot_residual_grid(comp, names, lim=None, pct=99.5, col_width=2.4, save=True,
                       font='Times New Roman', tag='', as_percent=None):
    scale, as_percent = _unit_scale(as_percent)
    unit_txt = '% of full scale' if as_percent else 'data units'
    file_tag = tag + ('_pct' if as_percent else '_abs')

    nz, ny, nx = gt['shape']
    zi, yi, xi = nz // 2, ny // 2, nx // 2
    n = len(names)

    resid = {k: scale * (volumes[k][comp] - gt[comp]) for k in names}
    if lim is None:   # per-component fallback, already in displayed units
        lim = max(np.nanpercentile(np.abs(r), pct) for r in resid.values())

    row_titles = [f'XY (z={zi})', f'XZ (y={yi})', f'YZ (x={xi})']

    with plt.rc_context({'font.family': font}):
        fig = plt.figure(figsize=(col_width * n + 0.8, 7.5))
        gs = fig.add_gridspec(
            4, n,
            height_ratios=[1, 0.45, 0.45, 0.07],
            hspace=0.12, wspace=0.05,
        )

        for j, k in enumerate(names):
            r = resid[k]
            slices = [r[zi, :, :], r[:, yi, :], r[:, :, xi]]
            for i, s in enumerate(slices):
                ax = fig.add_subplot(gs[i, j])
                im = ax.imshow(s, origin='lower', cmap='RdBu_r',
                               vmin=-lim, vmax=lim,
                               aspect='equal' if i == 0 else 'auto')
                ax.set_xticks([])
                ax.set_yticks([])
                if j == 0:
                    ax.set_ylabel(row_titles[i], fontsize=13, fontweight='bold')
                if i == 0:
                    rmse = np.sqrt(np.nanmean(r**2))
                    rmse_txt = f'{rmse:.2f}%' if as_percent else f'{rmse:.3g}'
                    ax.set_title(f'{k}\nRMSE = {rmse_txt}', fontsize=11,
                                 fontweight='bold')

        c0, c1 = int(n * 0.2), max(int(np.ceil(n * 0.8)), int(n * 0.2) + 1)
        cax = fig.add_subplot(gs[3, c0:c1])
        cbar = fig.colorbar(im, cax=cax, orientation='horizontal')
        cbar.set_label(f'{comp} residual (recon − GT), {unit_txt}',
                       fontsize=13, fontweight='bold')
        cbar.ax.tick_params(labelsize=11)

        if save:
            fig.savefig(os.path.join(out_dir, f'residual_{comp}_rows_xyz{file_tag}.png'),
                        dpi=300, bbox_inches='tight')
        plt.show()


# ---- run ----
shared_lim = global_residual_lim(recon_names)
print(f"Shared residual limit for Mx/My/Mz: ±{shared_lim:.4g} "
      f"{'%' if AS_PERCENT else '(data units)'}")

for comp in ['Mx', 'My', 'Mz']:
    plot_residual_grid(comp, recon_names, lim=shared_lim, tag='_shared')

plot_residual_grid('magnitude', recon_names)


In [ ]:
RECON_MIN_FRAC = 0.01   # angle only evaluated where |recon| > 1% of its own max

def stack_vec(v):
    return np.stack([v['Mx'], v['My'], v['Mz']], axis=-1)   # (nz, ny, nx, 3)

gt_vec = stack_vec(gt)

def fit_scale(recon_vec, region=mask):
    """Scalar s minimizing ||s*recon - GT||^2 over region, all 3 components jointly."""
    r, t = recon_vec[region], gt_vec[region]
    return np.nansum(r * t) / np.nansum(r * r)

scales = {n: fit_scale(stack_vec(volumes[n])) for n in recon_names}
for n, s in scales.items():
    print(f"{n:25s} scale factor = {s:.4g}" + ("   <-- NEGATIVE, check sign" if s <= 0 else ""))

def get_volume(name, rescaled=False):
    """Raw volume, or one with every component multiplied by the fitted scale."""
    v = volumes[name]
    if not rescaled:
        return v
    s = scales[name]
    return {c: (abs(s) if c == 'magnitude' else s) * v[c] for c in components}

def vector_metrics(recon, region=mask):
    """RMS of the full vector difference |M_recon - M_gt| inside the mask."""
    d2 = np.sum((stack_vec(recon) - gt_vec) ** 2, axis=-1)[region]
    rmse = np.sqrt(np.nanmean(d2))
    return {'vec_rmse': rmse, 'vec_nrmse_pct': 100 * rmse / ref_scale}

def angular_error_deg(recon, min_frac=RECON_MIN_FRAC):
    """Angle (deg, 0-180) between recon and GT vectors; NaN outside the valid region."""
    rv = stack_vec(recon)
    rmag = np.linalg.norm(rv, axis=-1)
    gmag = np.linalg.norm(gt_vec, axis=-1)
    valid = mask & (rmag > min_frac * np.nanmax(rmag))
    denom = np.where(valid, rmag * gmag, 1.0)
    cos = np.sum(rv * gt_vec, axis=-1) / denom
    ang = np.degrees(np.arccos(np.clip(cos, -1, 1)))
    ang[~valid] = np.nan
    return ang

ang = {n: angular_error_deg(volumes[n]) for n in recon_names}

In [ ]:
rows = []
for n in recon_names:
    row = {'volume': n, 'spacing': spacings[n], 'scale_factor': scales[n]}
    for rs in (False, True):
        tag = 'rescaled' if rs else 'raw'
        v = get_volume(n, rs)
        for c in components:
            row[f'{c}_nrmse_pct_{tag}'] = error_metrics(v[c], gt[c], mask)['nrmse_pct']
        row[f'vec_nrmse_pct_{tag}'] = vector_metrics(v)['vec_nrmse_pct']
    a = ang[n][np.isfinite(ang[n])]
    row.update(ang_mean_deg=a.mean(), ang_median_deg=np.median(a),
               ang_p95_deg=np.percentile(a, 95),
               ang_valid_frac=a.size / mask.sum())
    rows.append(row)

summary2 = pd.DataFrame(rows).set_index('volume')
summary2.to_csv(os.path.join(out_dir, 'metrics_raw_vs_rescaled_vector.csv'))
print(summary2.drop(columns='spacing').round(3).T)

In [ ]:
def _draw_grid(maps, names, cmap, vmin, vmax, cbar_label, title_fn, fname,
               col_width=2.4, font='Times New Roman', save=True):
    nz, ny, nx = gt['shape']
    zi, yi, xi = nz // 2, ny // 2, nx // 2
    n = len(names)
    row_titles = [f'XY (z={zi})', f'XZ (y={yi})', f'YZ (x={xi})']

    cm = plt.get_cmap(cmap).copy()
    cm.set_bad('lightgray')          # NaN (outside mask) shows as gray

    with plt.rc_context({'font.family': font}):
        fig = plt.figure(figsize=(col_width * n + 0.8, 7.5))
        gs = fig.add_gridspec(4, n, height_ratios=[1, 0.45, 0.45, 0.07],
                              hspace=0.12, wspace=0.05)
        for j, k in enumerate(names):
            m = maps[k]
            slices = [m[zi, :, :], m[:, yi, :], m[:, :, xi]]
            for i, s in enumerate(slices):
                ax = fig.add_subplot(gs[i, j])
                im = ax.imshow(s, origin='lower', cmap=cm, vmin=vmin, vmax=vmax,
                               aspect='equal' if i == 0 else 'auto')
                ax.set_xticks([]); ax.set_yticks([])
                if j == 0:
                    ax.set_ylabel(row_titles[i], fontsize=13, fontweight='bold')
                if i == 0:
                    ax.set_title(title_fn(k, m), fontsize=11, fontweight='bold')

        c0, c1 = int(n * 0.2), max(int(np.ceil(n * 0.8)), int(n * 0.2) + 1)
        cax = fig.add_subplot(gs[3, c0:c1])
        cbar = fig.colorbar(im, cax=cax, orientation='horizontal')
        cbar.set_label(cbar_label, fontsize=13, fontweight='bold')
        cbar.ax.tick_params(labelsize=11)
        if save:
            fig.savefig(os.path.join(out_dir, fname), dpi=300, bbox_inches='tight')
        plt.show()


def global_residual_lim(names, comps=('Mx', 'My', 'Mz'), pct=99.5,
                        as_percent=None, rescale=False):
    scale, _ = _unit_scale(as_percent)
    return scale * max(
        np.nanpercentile(np.abs(get_volume(k, rescale)[c] - gt[c]), pct)
        for c in comps for k in names
    )


def plot_residual_grid(comp, names, lim=None, pct=99.5, rescale=False,
                       as_percent=None, tag='', **kw):
    scale, as_percent = _unit_scale(as_percent)
    unit_txt = '% of full scale' if as_percent else 'data units'

    resid = {k: scale * (get_volume(k, rescale)[comp] - gt[comp]) for k in names}
    if lim is None:
        lim = max(np.nanpercentile(np.abs(r), pct) for r in resid.values())

    def title_fn(k, r):
        rmse = np.sqrt(np.nanmean(r ** 2))
        s_txt = f' (s={scales[k]:.2f})' if rescale else ''
        rmse_txt = f'{rmse:.2f}%' if as_percent else f'{rmse:.3g}'
        return f'{k}{s_txt}\nRMSE = {rmse_txt}'

    recon_txt = 'scaled recon' if rescale else 'recon'
    fname = (f"residual_{comp}_rows_xyz{tag}_{'rescaled' if rescale else 'raw'}"
             f"{'_pct' if as_percent else '_abs'}.png")
    _draw_grid(resid, names, 'RdBu_r', -lim, lim,
               f'{comp} residual ({recon_txt} − GT), {unit_txt}',
               title_fn, fname, **kw)


def plot_angular_grid(names, vmax=None, pct=99.5, **kw):
    maps = {k: ang[k] for k in names}
    if vmax is None:
        vmax = max(np.nanpercentile(m, pct) for m in maps.values())
    _draw_grid(maps, names, 'magma', 0, vmax, 'Angular error vs GT (degrees)',
               lambda k, m: f'{k}\nmean = {np.nanmean(m):.1f}°',
               'angular_error_rows_xyz.png', **kw)


# ---- run ----
# Use the RAW limit for both modes so raw vs. rescaled panels are directly comparable.
lim_shared = global_residual_lim(recon_names, rescale=False)
print(f"Shared residual limit: ±{lim_shared:.4g} {'%' if AS_PERCENT else '(data units)'}")

for rescale in (False, True):
    for comp in ['Mx', 'My', 'Mz']:
        plot_residual_grid(comp, recon_names, lim=lim_shared, rescale=rescale, tag='_shared')
    plot_residual_grid('magnitude', recon_names, rescale=rescale)

plot_angular_grid(recon_names)

In [ ]:
def unit_resid(recon, min_frac=RECON_MIN_FRAC):
    """Unit-vector residual (recon_hat - gt_hat), shape (nz,ny,nx,3). NaN outside valid region."""
    rv = stack_vec(recon)
    rmag = np.linalg.norm(rv, axis=-1)
    gmag = np.linalg.norm(gt_vec, axis=-1)
    valid = mask & (rmag > min_frac * np.nanmax(rmag))
    r_hat = rv / np.where(valid, rmag, 1.0)[..., None]
    g_hat = gt_vec / np.where(valid, gmag, 1.0)[..., None]
    out = r_hat - g_hat
    out[~valid] = np.nan
    return out

ures = {n: unit_resid(volumes[n]) for n in recon_names}

# Table: RMS and signed bias of each unit-vector component
rows = []
for n in recon_names:
    row = {'volume': n}
    for i, c in enumerate(['Mx', 'My', 'Mz']):
        d = ures[n][..., i]
        row[f'u{c}_rms'] = np.sqrt(np.nanmean(d**2))
        row[f'u{c}_bias'] = np.nanmean(d)
    rows.append(row)
unit_df = pd.DataFrame(rows).set_index('volume')
unit_df.to_csv(os.path.join(out_dir, 'unit_vector_component_errors.csv'))
print(unit_df.round(3))

# Plots: one figure per component, same layout as before
def plot_unit_resid_grid(comp, names, lim=None, pct=99.5, **kw):
    i = ['Mx', 'My', 'Mz'].index(comp)
    maps = {k: ures[k][..., i] for k in names}
    if lim is None:
        lim = max(np.nanpercentile(np.abs(m), pct) for m in maps.values())
    _draw_grid(maps, names, 'RdBu_r', -lim, lim,
               f'unit {comp} residual (recon − GT, direction cosine)',
               lambda k, m: f'{k}\nRMS = {np.sqrt(np.nanmean(m**2)):.3f}',
               f'unit_residual_{comp}_rows_xyz.png', **kw)

# Shared scale across components so Mx leakage is comparable to My/Mz error
lim_u = max(np.nanpercentile(np.abs(ures[k][..., i]), 99.5)
            for k in recon_names for i in range(3))
for comp in ['Mx', 'My', 'Mz']:
    plot_unit_resid_grid(comp, recon_names, lim=lim_u)